# Phase 10 v2: Competition Notebook

**Goal**: Beat 0.95081 (1st place) on Kaggle T4x2

## Strategy:
1. **ModernBERT-base** + **TF-IDF SVC** ensemble (transformer + n-gram diversity)
2. **5-fold** stratified CV (1920 train/fold → +20% DeepSeek per fold)
3. **R-Drop** regularization (bidirectional KL, kl_weight=0.3)
4. **EMA** weight averaging (decay=0.9995) for smoother, flatter minima
5. **Full-data model** (7 epochs, all 2400 samples) blended at α=0.6
6. **Nelder-Mead** ensemble weight optimization (6 initializations)
7. **LDAM + gradual DRW** (10× cap) + label smoothing (ε=0.1)
8. Conservative threshold nudge [0.85, 1.20] optimised on blended OOF

In [ ]:
# ============================================================
# FAST COMPETITION SETUP
# ============================================================
import os, warnings, gc, time, json, re, string
import numpy as np
import pandas as pd
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_scheduler

from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
NUM_LABELS = 6
LABEL_MAP = {0: 'Human', 1: 'DeepSeek', 2: 'Grok', 3: 'Claude', 4: 'Gemini', 5: 'ChatGPT'}

np.random.seed(SEED)
torch.manual_seed(SEED)

# ---- DEVICE SETUP ----
if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    torch.cuda.manual_seed_all(SEED)
    N_GPUS  = torch.cuda.device_count()
    USE_AMP = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print(f'GPU: {torch.cuda.get_device_name(0)} x {N_GPUS}')
    print(f'AMP: {USE_AMP} | TF32: disabled (prevents NaN)')
else:
    DEVICE = torch.device('cpu')
    N_GPUS  = 1
    USE_AMP = False

# ---- DATA PATHS ----
KAGGLE_PATHS = [
    '/kaggle/input/malto-recruitment-hackathon',
    '/kaggle/input/test-and-train',
    '/kaggle/input/datasets/aliivaezii/test-and-train',
    '/kaggle/input',
    '.', '..'
]

DATA_DIR = None
for p in KAGGLE_PATHS:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA_DIR = p
        break

# Fallback: recursive walk under /kaggle/input/ (handles any nesting depth)
if DATA_DIR is None and os.path.isdir('/kaggle/input'):
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'train.csv' in files:
            DATA_DIR = root
            break

assert DATA_DIR is not None, (
    'train.csv not found! Check dataset paths. '
    f'Searched: {KAGGLE_PATHS} + /kaggle/input/**/'
)

train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

train_df['TEXT'] = train_df['TEXT'].fillna("empty")
test_df['TEXT']  = test_df['TEXT'].fillna("empty")

# Detect ID column (competition CSVs vary)
ID_COL = 'ID' if 'ID' in test_df.columns else ('id' if 'id' in test_df.columns else None)
print(f'Data dir: {DATA_DIR}')
print(f'Test columns: {list(test_df.columns)}  |  ID_COL={ID_COL}')

y_all        = train_df['LABEL'].values
texts_train  = train_df['TEXT'].values
texts_test   = test_df['TEXT'].values

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('submissions', exist_ok=True)

print(f'Device: {DEVICE} | GPUs: {N_GPUS} | AMP: {USE_AMP}')
print(f'Train: {len(train_df)}, Test: {len(test_df)}')

In [ ]:
# ============================================================
# CONFIG
# ============================================================
MODEL_NAME = 'answerdotai/ModernBERT-base'
MAX_LEN     = 512

# Single-GPU training — DataParallel breaks gradient checkpointing
# (use_reentrant=True silently drops gradients for some params → val_f1 stays ~0.1)
BATCH_SIZE  = 4              # per-step micro-batch (small for T4 15GB + R-Drop 2× fwd)
GRAD_ACCUM  = 8              # effective batch = 4 × 8 = 32
EPOCHS      = 10
LR          = 3e-5
PATIENCE    = 3
N_FOLDS     = 5
NUM_WORKERS = 4

print(f'Model: {MODEL_NAME}')
print(f'Batch: {BATCH_SIZE} x {GRAD_ACCUM} = {BATCH_SIZE * GRAD_ACCUM}  (single GPU + grad checkpointing)')
print(f'Folds: {N_FOLDS}, Epochs: {EPOCHS}, Patience: {PATIENCE}')
print(f'Single GPU mode (no DataParallel) + gradient checkpointing')

In [ ]:
# ============================================================
# LOSS FUNCTIONS
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        logits = logits.float()
        weight = self.alpha.to(logits.device) if self.alpha is not None else None
        ce  = F.cross_entropy(logits, targets, weight=weight, reduction='none')
        pt  = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class LDAMLoss(nn.Module):
    """LDAM + gradual DRW + label smoothing.

    Two key anti-overfitting measures beyond the previous version:

    1. DRW cap lowered 20× → 10×
       The previous 20× cap gave DeepSeek a DRW weight of 1.93×. With only 80
       DeepSeek training samples per fold (~53 in the training split), this is
       so large that the model memorises those samples rather than learning
       generalisable features. 10× cap → DeepSeek weight ~1.0 (proportional
       to its actual frequency × 6). Still upweights minority, but not so much
       that it overfits.

    2. Label smoothing ε = 0.1
       Prevents the model from being overconfident on minority classes.
       When OOF DeepSeek F1 = 0.78 but public score implies ≈ 0.54, the model
       is overconfident on training-distribution DeepSeek, and label smoothing
       explicitly regularises this. Also helps calibration (reduces the
       abnormally high temperature T = 1.55 that we saw).
    """
    def __init__(self, class_counts, max_margin=0.5,
                 drw_epoch=1, drw_ramp_epochs=3, label_smoothing=0.1):
        super().__init__()
        safe = np.maximum(class_counts, 1).astype(np.float64)

        margins = np.clip(max_margin / np.power(safe, 0.25), 0, 1.0)
        self.register_buffer('margins', torch.tensor(margins, dtype=torch.float32))

        self.drw_epoch       = drw_epoch
        self.drw_ramp_epochs = drw_ramp_epochs
        self.current_epoch   = 0
        self.label_smoothing = label_smoothing

        # DRW target weights – 10× cap (was 20×; reduces DeepSeek memorisation)
        inv  = 1.0 / safe
        w    = inv / inv.sum() * len(class_counts)
        w    = np.clip(w, w.min(), w.min() * 10)
        w    = w / w.sum() * len(class_counts)
        w    = np.nan_to_num(w, nan=1.0, posinf=1.0, neginf=1.0)
        self.register_buffer('drw_weights', torch.tensor(w, dtype=torch.float32))

    def set_epoch(self, epoch):
        self.current_epoch = epoch

    def _blended_weight(self, device):
        ep = self.current_epoch
        if ep < self.drw_epoch:
            return None
        t    = min((ep - self.drw_epoch) / max(self.drw_ramp_epochs, 1), 1.0)
        ones = torch.ones_like(self.drw_weights)
        return (ones + t * (self.drw_weights - ones)).to(device)

    def forward(self, logits, targets):
        logits = logits.float()
        if torch.isnan(logits).any():
            logits = torch.nan_to_num(logits, nan=0.0)

        margins  = self.margins.to(logits.device)[targets]
        adjusted = logits.clone()
        adjusted[torch.arange(len(targets), device=logits.device), targets] -= margins

        weight = self._blended_weight(logits.device)
        loss   = F.cross_entropy(adjusted, targets, weight=weight,
                                  label_smoothing=self.label_smoothing)

        if torch.isnan(loss):
            return F.cross_entropy(logits, targets)
        return loss


def rdrop_kl(logits1, logits2, kl_weight=0.3):
    """Bidirectional KL divergence for R-Drop regularization.

    WHY:  The same input passed twice through a model with dropout produces two
    different predictions.  R-Drop penalises disagreement between these two
    distributions, forcing the model to produce dropout-invariant (i.e. more
    robust) representations.  This is especially helpful for minority classes
    (DeepSeek, Claude) where the model tends to memorise training-set patterns
    rather than learning transferable features.

    kl_weight=0.3 is a safe default (original paper used 0.5; 0.3 balances
    regularisation strength against training speed since we run 2 forward passes
    per step).
    """
    p  = F.log_softmax(logits1.float(), dim=-1)
    q  = F.softmax(logits2.float(),     dim=-1)
    q_ = F.log_softmax(logits2.float(), dim=-1)
    p_ = F.softmax(logits1.float(),     dim=-1)
    kl = (F.kl_div(p, q, reduction='batchmean') +
          F.kl_div(q_, p_, reduction='batchmean')) * 0.5
    return kl_weight * kl


print('Loss functions ready.')
print('  LDAMLoss:  margins + gradual DRW (10× cap) + label_smoothing=0.1')
print('  rdrop_kl:  bidirectional KL (kl_weight=0.3) for R-Drop regularization')

In [ ]:
# ============================================================
# TOKENIZATION
# ============================================================
print('Tokenizing...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_enc = tokenizer(
    list(texts_train), padding='max_length', truncation=True,
    max_length=MAX_LEN, return_tensors='pt'
)
test_enc = tokenizer(
    list(texts_test), padding='max_length', truncation=True,
    max_length=MAX_LEN, return_tensors='pt'
)

print(f'Train: {train_enc["input_ids"].shape}, Test: {test_enc["input_ids"].shape}')

In [ ]:
# ============================================================
# DATASET & OPTIMIZER
# ============================================================
class TextDataset(Dataset):
    def __init__(self, encodings, labels=None, indices=None):
        self.encodings = encodings
        self.labels = labels
        self.indices = indices

    def __len__(self):
        return len(self.indices) if self.indices is not None else len(self.encodings['input_ids'])

    def __getitem__(self, idx):
        real_idx = self.indices[idx] if self.indices is not None else idx
        item = {k: v[real_idx] for k, v in self.encodings.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[real_idx], dtype=torch.long)
        return item


def get_optimizer(model, lr, llrd=0.9):
    """Optimizer with layer-wise learning rate decay.
    
    Uses precise layer-name matching for ModernBERT ('layers.N.' pattern).
    Phase 10 originally used f'.{i}.' which is too broad and picks up wrong params.
    """
    import re as _re
    no_decay = ['bias', 'LayerNorm', 'layernorm', 'layer_norm']
    params = []

    # Classifier head: full LR
    head = [(n, p) for n, p in model.named_parameters()
            if 'classifier' in n or 'head' in n]
    if head:
        params.append({'params': [p for n, p in head if not any(nd in n for nd in no_decay)], 'lr': lr, 'weight_decay': 0.01})
        params.append({'params': [p for n, p in head if any(nd in n for nd in no_decay)],     'lr': lr, 'weight_decay': 0.0})

    # Detect layer count from parameter names (works for both DeBERTa and ModernBERT)
    all_names = [n for n, _ in model.named_parameters()]
    layer_nums = set()
    for n in all_names:
        m = _re.search(r'(?:encoder\.layer|layers)\.(\d+)\.', n)
        if m:
            layer_nums.add(int(m.group(1)))
    num_layers = max(layer_nums) + 1 if layer_nums else 12

    # Transformer layers: decaying LR (deeper layers get smaller LR)
    for i in range(num_layers - 1, -1, -1):
        layer_lr = lr * (llrd ** (num_layers - 1 - i))
        # Precise patterns – avoids accidentally matching 'layers.10.' when searching for '.0.'
        patterns = [f'encoder.layer.{i}.', f'layers.{i}.']
        layer_params = [(n, p) for n, p in model.named_parameters()
                        if any(pat in n for pat in patterns)]
        if layer_params:
            wd_p   = [p for n, p in layer_params if not any(nd in n for nd in no_decay)]
            nowd_p = [p for n, p in layer_params if any(nd in n for nd in no_decay)]
            if wd_p:   params.append({'params': wd_p,   'lr': layer_lr, 'weight_decay': 0.01})
            if nowd_p: params.append({'params': nowd_p, 'lr': layer_lr, 'weight_decay': 0.0})

    # Embeddings: smallest LR
    emb_lr = lr * (llrd ** num_layers)
    emb_params = [(n, p) for n, p in model.named_parameters() if 'embed' in n.lower()]
    if emb_params:
        wd_p   = [p for n, p in emb_params if not any(nd in n for nd in no_decay)]
        nowd_p = [p for n, p in emb_params if any(nd in n for nd in no_decay)]
        if wd_p:   params.append({'params': wd_p,   'lr': emb_lr, 'weight_decay': 0.01})
        if nowd_p: params.append({'params': nowd_p, 'lr': emb_lr, 'weight_decay': 0.0})

    # Catch any remaining params
    assigned = set(id(p) for grp in params for p in grp['params'])
    rem = [p for n, p in model.named_parameters() if id(p) not in assigned and p.requires_grad]
    if rem:
        params.append({'params': rem, 'lr': lr * 0.5, 'weight_decay': 0.01})

    return torch.optim.AdamW(params)

print('Dataset & optimizer ready (precise ModernBERT layer matching).')

In [ ]:
# ============================================================
# FAST K-FOLD TRAINING  (LDAM + gradual DRW + R-Drop + EMA)
# ============================================================

def _ema_load(model, ema_state):
    """Load EMA state into model, respecting original dtypes for each tensor."""
    target_sd = model.state_dict()
    load_dict = {}
    for k, v in ema_state.items():
        orig = target_sd[k]
        if orig.is_floating_point():
            load_dict[k] = v.to(dtype=orig.dtype, device=orig.device)
        else:
            load_dict[k] = orig
    model.load_state_dict(load_dict)


def train_fold(fold_idx, train_idx, val_idx):
    gc.collect()
    torch.cuda.empty_cache()

    print(f'\n{"="*50}')
    print(f'FOLD {fold_idx + 1}/{N_FOLDS} | Train: {len(train_idx)}, Val: {len(val_idx)}')
    print(f'{"="*50}')

    train_loader = DataLoader(
        TextDataset(train_enc, y_all, train_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=True, drop_last=True
    )
    val_loader = DataLoader(
        TextDataset(train_enc, y_all, val_idx),
        batch_size=BATCH_SIZE * 4, num_workers=NUM_WORKERS, pin_memory=True
    )
    test_loader = DataLoader(
        TextDataset(test_enc),
        batch_size=BATCH_SIZE * 4, num_workers=NUM_WORKERS, pin_memory=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
    )
    # Gradient checkpointing with use_reentrant=False (safe with autograd)
    model.gradient_checkpointing_enable(
        gradient_checkpointing_kwargs={"use_reentrant": False}
    )
    model.to(DEVICE)
    # No DataParallel — it breaks gradient checkpointing on T4×2

    fold_counts = np.bincount(y_all[train_idx], minlength=NUM_LABELS).astype(float)
    criterion   = LDAMLoss(fold_counts, max_margin=0.5, drw_epoch=1, drw_ramp_epochs=3)
    criterion.to(DEVICE)

    drw_w   = criterion.drw_weights.cpu().numpy().round(2)
    margins = criterion.margins.cpu().numpy().round(3)
    print(f'  LDAM margins:            {margins}')
    print(f'  DRW target weights:      {drw_w}')
    print(f'  DRW ramp: E1→33% E2→67% E3→100%')
    print(f'  R-Drop kl=0.3 (sequential) | EMA=0.9995 | grad_accum={GRAD_ACCUM}')
    print(f'  Gradient checkpointing: ON (use_reentrant=False)')

    optimizer = get_optimizer(model, lr=LR, llrd=0.9)

    total_opt_steps = (len(train_loader) // GRAD_ACCUM) * EPOCHS
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_opt_steps),
        num_training_steps=total_opt_steps)

    scaler = GradScaler(enabled=USE_AMP)

    EMA_DECAY = 0.9995
    ema_state = {k: v.cpu().clone().float() if v.is_floating_point() else v.cpu().clone()
                 for k, v in model.state_dict().items()}

    best_f1          = 0
    best_ema_state   = None
    patience_counter = 0

    for epoch in range(EPOCHS):
        t0 = time.time()
        criterion.set_epoch(epoch)

        if epoch < criterion.drw_epoch:
            drw_pct = '  0%'
        else:
            t = min((epoch - criterion.drw_epoch) / criterion.drw_ramp_epochs, 1.0)
            drw_pct = f'{int(t*100):3d}%'

        # ── Train ────────────────────────────────────────────
        model.train()
        total_loss  = 0.0
        valid_steps = 0
        optimizer.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f'E{epoch+1}', leave=False)):
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')

            # R-Drop: SEQUENTIAL backward — only 1 graph in VRAM at a time
            # Pass 1: forward → backward → free graph, keep logits detached
            with autocast(enabled=USE_AMP):
                out1  = model(**inputs)
                loss1 = criterion(out1.logits, labels) * 0.5 / GRAD_ACCUM
                logits1_det = out1.logits.detach()
                loss1_val = loss1.item()

            if not (torch.isnan(loss1) or torch.isinf(loss1)):
                scaler.scale(loss1).backward()
            del out1, loss1

            # Pass 2: forward → KL with detached logits1 → backward
            with autocast(enabled=USE_AMP):
                out2  = model(**inputs)
                loss2 = criterion(out2.logits, labels) * 0.5 / GRAD_ACCUM
                kl    = rdrop_kl(logits1_det, out2.logits, kl_weight=0.3) / GRAD_ACCUM
                loss_p2 = loss2 + kl
                loss_p2_val = loss_p2.item()

            if not (torch.isnan(loss_p2) or torch.isinf(loss_p2)):
                scaler.scale(loss_p2).backward()
            del out2, logits1_det, loss_p2

            # Track total loss from both passes
            total_loss  += (loss1_val + loss_p2_val) * GRAD_ACCUM
            valid_steps += 1

            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

                with torch.no_grad():
                    for k, v in model.state_dict().items():
                        if ema_state[k].is_floating_point():
                            ema_state[k].mul_(EMA_DECAY).add_(v.cpu().float(),
                                                               alpha=1.0 - EMA_DECAY)

        avg_loss = total_loss / max(valid_steps, 1)

        # ── Validate with EMA weights ─────────────────────────
        orig_state_tmp = {k: v.clone() for k, v in model.state_dict().items()}
        _ema_load(model, ema_state)

        model.eval()
        preds, labels_list = [], []
        with torch.no_grad():
            for batch in val_loader:
                inputs = {k: v.to(DEVICE) for k, v in batch.items()}
                labels = inputs.pop('labels')
                with autocast(enabled=USE_AMP):
                    outputs = model(**inputs)
                preds.extend(outputs.logits.argmax(-1).cpu().numpy())
                labels_list.extend(labels.cpu().numpy())

        model.load_state_dict(orig_state_tmp)

        val_f1  = f1_score(labels_list, preds, average='macro')
        elapsed = time.time() - t0

        if val_f1 > best_f1:
            best_f1        = val_f1
            best_ema_state = {k: v.clone() for k, v in ema_state.items()}
            patience_counter = 0
            marker = ' ** BEST **'
        else:
            patience_counter += 1
            marker = f' (p={patience_counter}/{PATIENCE})'

        print(f'  E{epoch+1} | loss={avg_loss:.4f} | val_f1(EMA)={val_f1:.4f} '
              f'| DRW={drw_pct} | {elapsed:.0f}s{marker}')

        if patience_counter >= PATIENCE:
            print(f'  Early stop at E{epoch+1}')
            break

    # ── Final inference with best EMA checkpoint ──────────────
    _ema_load(model, best_ema_state)
    model.eval()

    val_logits = []
    with torch.no_grad():
        for batch in val_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            inputs.pop('labels', None)
            with autocast(enabled=USE_AMP):
                outputs = model(**inputs)
            val_logits.extend(outputs.logits.float().cpu().numpy())

    test_logits = []
    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=USE_AMP):
                outputs = model(**inputs)
            test_logits.extend(outputs.logits.float().cpu().numpy())

    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

    return np.array(val_logits), np.array(test_logits), best_f1

print('Training function ready.')
print(f'  LDAM + DRW | R-Drop sequential (kl=0.3) | EMA (0.9995) | grad_accum={GRAD_ACCUM}')
print(f'  Single GPU + gradient checkpointing (use_reentrant=False)')

In [ ]:
# ============================================================
# RUN K-FOLD TRAINING
# ============================================================
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_logits = np.zeros((len(y_all), NUM_LABELS))
test_logits_sum = np.zeros((len(test_df), NUM_LABELS))
fold_scores = []

t_start = time.time()

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(y_all)), y_all)):
    val_logits, test_logits, best_f1 = train_fold(fold_idx, train_idx, val_idx)
    oof_logits[val_idx] = val_logits
    test_logits_sum += test_logits
    fold_scores.append(best_f1)
    print(f'  Fold {fold_idx+1} best: {best_f1:.4f} | Total time: {(time.time()-t_start)/60:.1f}min')

test_logits_avg = test_logits_sum / N_FOLDS

print(f'\n{"="*50}')
print(f'{N_FOLDS}-FOLD COMPLETE')
print(f'  Scores: {[f"{s:.4f}" for s in fold_scores]}')
print(f'  Mean: {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')
print(f'  Time: {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')

In [ ]:
# ============================================================
# TEMPERATURE SCALING
# ============================================================
logits_t = torch.tensor(oof_logits, dtype=torch.float32)
labels_t = torch.tensor(y_all, dtype=torch.long)

best_temp = 1.0
best_nll = float('inf')
for temp in np.arange(0.3, 3.0, 0.05):
    nll = F.cross_entropy(logits_t / temp, labels_t).item()
    if nll < best_nll:
        best_nll = nll
        best_temp = temp

print(f'Optimal temperature: {best_temp:.2f}')

oof_probs = torch.softmax(logits_t / best_temp, dim=-1).numpy()
test_probs = torch.softmax(torch.tensor(test_logits_avg) / best_temp, dim=-1).numpy()

oof_f1 = f1_score(y_all, oof_probs.argmax(1), average='macro')
print(f'OOF F1: {oof_f1:.4f}')

print('\nPer-class report:')
print(classification_report(y_all, oof_probs.argmax(1),
    target_names=[f'{v}' for v in LABEL_MAP.values()]))

In [ ]:
# ============================================================
# CLASSICAL MODEL: TF-IDF + Calibrated SVC  (cheap diversity)
# ============================================================
# WHY:  SVC captures char/word n-gram patterns that transformers miss.
#       Phase 9 showed SVC OOF F1 ≈ 0.92–0.93.  Even a 10–15% blend weight
#       can flip borderline minority-class predictions.  Trains in ~30 sec.
#
# SAME FOLD SPLITS as ModernBERT (same StratifiedKFold params) → OOFs are
# directly comparable and can be blended row-by-row.

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from scipy.sparse import hstack as sparse_hstack

print('Building TF-IDF features...')
t_svc = time.time()

# Character n-grams (3–5 chars, word-boundary aware)
char_tfidf = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5),
    max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True
)
# Word n-grams (unigrams + bigrams)
word_tfidf = TfidfVectorizer(
    analyzer='word', ngram_range=(1, 2),
    max_features=50000, min_df=2, max_df=0.95, sublinear_tf=True
)

char_train = char_tfidf.fit_transform(texts_train)
char_test  = char_tfidf.transform(texts_test)
word_train = word_tfidf.fit_transform(texts_train)
word_test  = word_tfidf.transform(texts_test)

X_sparse_train = sparse_hstack([char_train, word_train])
X_sparse_test  = sparse_hstack([char_test,  word_test])
print(f'  Features: {X_sparse_train.shape[1]:,} (char={char_train.shape[1]:,} + word={word_train.shape[1]:,})')

# OOF predictions with SAME fold splits as ModernBERT
svc_oof_probs      = np.zeros((len(y_all), NUM_LABELS))
svc_test_probs_sum = np.zeros((len(texts_test), NUM_LABELS))

skf_svc = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold_idx, (train_idx, val_idx) in enumerate(skf_svc.split(np.zeros(len(y_all)), y_all)):
    svc = CalibratedClassifierCV(
        LinearSVC(C=5.0, class_weight='balanced', max_iter=20000),
        cv=3, method='sigmoid'
    )
    svc.fit(X_sparse_train[train_idx], y_all[train_idx])
    svc_oof_probs[val_idx] = svc.predict_proba(X_sparse_train[val_idx])
    svc_test_probs_sum    += svc.predict_proba(X_sparse_test)

    fold_f1 = f1_score(y_all[val_idx], svc_oof_probs[val_idx].argmax(1), average='macro')
    print(f'  SVC Fold {fold_idx+1}: F1={fold_f1:.4f}')

svc_test_probs = svc_test_probs_sum / N_FOLDS
svc_oof_f1     = f1_score(y_all, svc_oof_probs.argmax(1), average='macro')

print(f'\nSVC OOF F1: {svc_oof_f1:.4f}  ({time.time()-t_svc:.0f}s)')
print(classification_report(y_all, svc_oof_probs.argmax(1),
      target_names=[f'{v}' for v in LABEL_MAP.values()]))

In [ ]:
# ============================================================
# FULL-DATA MODEL  (trains on all 2400 samples)
# ============================================================
FULL_EPOCHS = 7

def train_full_data():
    gc.collect()
    torch.cuda.empty_cache()

    print(f'\n{"="*50}')
    print(f'FULL-DATA MODEL | All {len(y_all)} samples | {FULL_EPOCHS} epochs')
    print(f'{"="*50}')

    all_idx = np.arange(len(y_all))
    train_loader = DataLoader(
        TextDataset(train_enc, y_all, all_idx),
        batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
        pin_memory=True, drop_last=True
    )
    test_loader = DataLoader(
        TextDataset(test_enc),
        batch_size=BATCH_SIZE * 2, num_workers=NUM_WORKERS, pin_memory=True
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True
    )
    model.gradient_checkpointing_enable()
    if N_GPUS > 1:
        model = nn.DataParallel(model)
    model.to(DEVICE)

    fold_counts = np.bincount(y_all, minlength=NUM_LABELS).astype(float)
    criterion   = LDAMLoss(fold_counts, max_margin=0.5, drw_epoch=1,
                           drw_ramp_epochs=3, label_smoothing=0.1)
    criterion.to(DEVICE)

    drw_w   = criterion.drw_weights.cpu().numpy().round(2)
    margins = criterion.margins.cpu().numpy().round(3)
    print(f'  LDAM margins:       {margins}')
    print(f'  DRW target weights: {drw_w}')
    print(f'  R-Drop kl=0.3 (sequential) | EMA=0.9995 | grad_accum={GRAD_ACCUM}')
    print(f'  Gradient checkpointing: ON')

    actual_model = model.module if hasattr(model, 'module') else model
    optimizer    = get_optimizer(actual_model, lr=LR * 0.8, llrd=0.9)

    total_opt_steps = (len(train_loader) // GRAD_ACCUM) * FULL_EPOCHS
    scheduler = get_scheduler('cosine', optimizer,
        num_warmup_steps=int(0.1 * total_opt_steps),
        num_training_steps=total_opt_steps)

    scaler = GradScaler(enabled=USE_AMP)

    EMA_DECAY = 0.9995
    ema_state = {k: v.cpu().clone().float() if v.is_floating_point() else v.cpu().clone()
                 for k, v in actual_model.state_dict().items()}

    for epoch in range(FULL_EPOCHS):
        t0 = time.time()
        criterion.set_epoch(epoch)

        ep = epoch
        if ep < criterion.drw_epoch:
            drw_pct = '  0%'
        else:
            t = min((ep - criterion.drw_epoch) / criterion.drw_ramp_epochs, 1.0)
            drw_pct = f'{int(t*100):3d}%'

        model.train()
        total_loss  = 0.0
        valid_steps = 0
        optimizer.zero_grad()

        for step, batch in enumerate(tqdm(train_loader, desc=f'Full E{epoch+1}', leave=False)):
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = inputs.pop('labels')

            # Pass 1: forward → backward → free graph
            with autocast(enabled=USE_AMP):
                out1  = model(**inputs)
                loss1 = criterion(out1.logits, labels) * 0.5 / GRAD_ACCUM
                logits1_det = out1.logits.detach()
                loss1_val = loss1.item()

            if not (torch.isnan(loss1) or torch.isinf(loss1)):
                scaler.scale(loss1).backward()
            del out1, loss1

            # Pass 2: forward → KL with detached logits1 → backward
            with autocast(enabled=USE_AMP):
                out2  = model(**inputs)
                loss2 = criterion(out2.logits, labels) * 0.5 / GRAD_ACCUM
                kl    = rdrop_kl(logits1_det, out2.logits, kl_weight=0.3) / GRAD_ACCUM
                loss_p2 = loss2 + kl
                loss_p2_val = loss_p2.item()

            if not (torch.isnan(loss_p2) or torch.isinf(loss_p2)):
                scaler.scale(loss_p2).backward()
            del out2, logits1_det, loss_p2

            # Track total loss from both passes
            total_loss  += (loss1_val + loss_p2_val) * GRAD_ACCUM
            valid_steps += 1

            if (step + 1) % GRAD_ACCUM == 0 or (step + 1) == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

                cur = model.module if hasattr(model, 'module') else model
                with torch.no_grad():
                    for k, v in cur.state_dict().items():
                        if ema_state[k].is_floating_point():
                            ema_state[k].mul_(EMA_DECAY).add_(v.cpu().float(),
                                                               alpha=1.0 - EMA_DECAY)

        avg_loss = total_loss / max(valid_steps, 1)
        print(f'  Full E{epoch+1}/{FULL_EPOCHS} | loss={avg_loss:.4f} | DRW={drw_pct} | {time.time()-t0:.0f}s')

    cur_model = model.module if hasattr(model, 'module') else model
    _ema_load(cur_model, ema_state)

    model.eval()
    test_logits = []
    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(enabled=USE_AMP):
                outputs = model(**inputs)
            test_logits.extend(outputs.logits.float().cpu().numpy())

    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()

    return np.array(test_logits)


t_full = time.time()
full_test_logits = train_full_data()
print(f'\nFull-data training done in {(time.time()-t_full)/60:.1f} min')
print(f'Total elapsed:             {(time.time()-t_start)/60:.1f} min')

In [ ]:
# ============================================================
# ENSEMBLE: Nelder-Mead weighted blend  (ModernBERT + SVC)
# ============================================================
# Component 1: ModernBERT fold-average (temperature-scaled)  — has OOF
# Component 2: ModernBERT full-data    (temperature-scaled)  — NO OOF
# Component 3: SVC fold-average        (calibrated probs)    — has OOF
#
# We optimise [w_mbert, w_svc] on OOF (components 1 & 3).
# For test, ModernBERT = α × fold + (1-α) × full, then blended with SVC.

from scipy.optimize import minimize

print('Nelder-Mead ensemble weight optimization...')

# Temperature-scale the full-data model
full_test_probs = torch.softmax(
    torch.tensor(full_test_logits, dtype=torch.float32) / best_temp, dim=-1
).numpy()
print(f'  Temperature applied: {best_temp:.2f}  (calibrated on ModernBERT OOF)')

# ── Optimise on OOF (components with held-out predictions) ──
def neg_macro_f1(weights, probs_list, labels):
    w = np.abs(weights)
    w = w / w.sum()
    blended = sum(wi * pi for wi, pi in zip(w, probs_list))
    return -f1_score(labels, blended.argmax(1), average='macro')

oof_components = [oof_probs, svc_oof_probs]

best_result = None
best_neg_f1 = 0
for init in [[0.85, 0.15], [0.80, 0.20], [0.90, 0.10],
             [0.75, 0.25], [0.70, 0.30], [0.95, 0.05]]:
    result = minimize(neg_macro_f1, init, args=(oof_components, y_all),
                      method='Nelder-Mead',
                      options={'maxiter': 5000, 'xatol': 1e-5, 'fatol': 1e-6})
    if best_result is None or result.fun < best_neg_f1:
        best_neg_f1 = result.fun
        best_result = result

opt_w = np.abs(best_result.x) / np.abs(best_result.x).sum()

# OOF blend (for threshold optimization)
oof_blend   = opt_w[0] * oof_probs + opt_w[1] * svc_oof_probs
oof_blend_f1 = f1_score(y_all, oof_blend.argmax(1), average='macro')

print(f'\n  Optimal weights: ModernBERT={opt_w[0]:.3f}, SVC={opt_w[1]:.3f}')
print(f'  ModernBERT OOF F1: {oof_f1:.4f}')
print(f'  SVC OOF F1:        {svc_oof_f1:.4f}')
print(f'  Blended OOF F1:    {oof_blend_f1:.4f}  (Nelder-Mead optimised)')

# ── Build test blend ──────────────────────────────────────────
# ModernBERT test = α × fold_avg + (1-α) × full_data
ALPHA = 0.6
mbert_test_blend = ALPHA * test_probs + (1 - ALPHA) * full_test_probs

# Final test blend: weighted combination of ModernBERT + SVC
test_blend = opt_w[0] * mbert_test_blend + opt_w[1] * svc_test_probs

print(f'\n  Test blend: {opt_w[0]:.0%} × ModernBERT ({ALPHA:.0%} fold + {1-ALPHA:.0%} full) '
      f'+ {opt_w[1]:.0%} × SVC')

# Per-class comparison
fold_preds  = test_probs.argmax(1)
full_preds  = full_test_probs.argmax(1)
svc_preds   = svc_test_probs.argmax(1)
blend_preds = test_blend.argmax(1)
print(f'\nTest prediction distribution:')
print(f'  {"Class":10s}  {"MBfold":>6}  {"MBfull":>6}  {"SVC":>6}  {"Blend":>6}')
for i, name in LABEL_MAP.items():
    print(f'  {name:10s}  {(fold_preds==i).sum():6d}  {(full_preds==i).sum():6d}'
          f'  {(svc_preds==i).sum():6d}  {(blend_preds==i).sum():6d}')

In [ ]:
# ============================================================
# CONSERVATIVE THRESHOLD NUDGE  (optimised on blended OOF)
# ============================================================
# Bounds [0.85, 1.20] prevent the Human=0.40 overfit from earlier runs.
# NOW optimised on oof_blend (ModernBERT + SVC) instead of ModernBERT alone.
# Steps=40, passes=5 for fine-grained search.

def conservative_threshold_sweep(oof_probs_2400, labels, n_classes=6,
                                  steps=40, passes=5):
    best_mult = np.ones(n_classes)
    best_f1   = f1_score(labels, oof_probs_2400.argmax(1), average='macro')
    lo, hi    = 0.85, 1.20
    for _ in range(passes):
        improved = False
        for cls in range(n_classes):
            for m in np.linspace(lo, hi, steps):
                trial = best_mult.copy()
                trial[cls] = m
                f1 = f1_score(labels, (oof_probs_2400 * trial).argmax(1), average='macro')
                if f1 > best_f1 + 1e-6:
                    best_f1, best_mult, improved = f1, trial.copy(), True
        if not improved:
            break
    return best_mult, best_f1

print('Conservative threshold nudge on blended OOF...')
thresholds, thresh_f1 = conservative_threshold_sweep(oof_blend, y_all)
print(f'OOF F1 after nudge: {thresh_f1:.4f}  (was {oof_blend_f1:.4f})')
print('Multipliers:', {LABEL_MAP[i]: round(thresholds[i], 3) for i in range(NUM_LABELS)})

# Apply thresholds to blended test probabilities
final_preds = (test_blend * thresholds).argmax(1)

print(f'\n{"="*50}')
print(f'PIPELINE SUMMARY')
print(f'  ModernBERT OOF F1:     {oof_f1:.4f}')
print(f'  SVC OOF F1:            {svc_oof_f1:.4f}')
print(f'  Blended OOF F1:        {oof_blend_f1:.4f}')
print(f'  + threshold nudge:     {thresh_f1:.4f}')
print(f'  Ensemble weights:      MB={opt_w[0]:.3f} SVC={opt_w[1]:.3f}')
print(f'  MB fold/full α:        {ALPHA:.1f}/{1-ALPHA:.1f}')
print(f'  Full model epochs:     {FULL_EPOCHS}')
print(f'  Temperature:           {best_temp:.2f}')
print(f'  Target (1st place):    0.9509')
print(f'{"="*50}')

In [ ]:
# ============================================================
# CREATE SUBMISSION
# ============================================================
id_col = (test_df['ID'] if 'ID' in test_df.columns
          else test_df['id'] if 'id' in test_df.columns
          else np.arange(len(test_df)))

submission = pd.DataFrame({'ID': id_col, 'LABEL': final_preds})
submission.to_csv('submission.csv', index=False)

print('Submission saved!')
print(f'\nPrediction distribution:')
for i, name in LABEL_MAP.items():
    count = (final_preds == i).sum()
    print(f'  {name}: {count} ({100*count/len(final_preds):.1f}%)')

print(f'\n{"="*50}')
print(f'FINAL SUMMARY')
print(f'  ModernBERT OOF F1:      {oof_f1:.4f}')
print(f'  SVC OOF F1:             {svc_oof_f1:.4f}')
print(f'  Blended OOF F1:         {oof_blend_f1:.4f}')
print(f'  + threshold nudge:      {thresh_f1:.4f}')
print(f'  Ensemble: MB={opt_w[0]:.3f} SVC={opt_w[1]:.3f}')
print(f'  MB blend: {ALPHA:.0%} fold + {1-ALPHA:.0%} full ({FULL_EPOCHS}ep)')
print(f'  Temperature:            {best_temp:.2f}')
print(f'  Target (1st place):     0.9509')
print(f'  Total time:             {(time.time()-t_start)/60:.1f} min')
print(f'{"="*50}')